In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, entropy, chi2_contingency
import joblib
import json
import os

# Cargar configuración
with open("../config.json") as f:
    config = json.load(f)

data_path = os.path.join("..",config["data"]["path"])
target = config["data"]["target_column"]

# Cargar datos actuales
df_actual = pd.read_csv(data_path)
df_actual["Vehicle_Age"] = 2025 - df_actual["Year"]
df_actual.drop(columns=["Year", "Car_Name"], inplace=True)

# Cargar modelo
model = joblib.load("../models/best_model.pkl")

# Cargar datos históricos
df_historico = pd.read_csv("data/historico.csv")

# Predicción actual
df_actual["prediction"] = model.predict(df_actual.drop(columns=[target]))

# Métricas de drift
def ks_test(col):
    return ks_2samp(df_historico[col], df_actual[col]).statistic

def psi(col, bins=10):
    hist1, _ = np.histogram(df_historico[col], bins=bins)
    hist2, _ = np.histogram(df_actual[col], bins=bins)
    hist1 = hist1 / sum(hist1)
    hist2 = hist2 / sum(hist2)
    return np.sum((hist1 - hist2) * np.log((hist1 + 1e-6) / (hist2 + 1e-6)))

def js_div(col):
    p = df_historico[col].value_counts(normalize=True)
    q = df_actual[col].value_counts(normalize=True)
    return entropy((p + q) / 2, p) + entropy((p + q) / 2, q)

def chi_square(col):
    table = pd.crosstab(df_historico[col], df_actual[col])
    return chi2_contingency(table)[0]

# Aplicar métricas
drift_report = {}
for col in df_actual.columns:
    if col == target or col == "prediction":
        continue
    if df_actual[col].dtype == "object":
        drift_report[col] = {
            "JS_divergence": js_div(col),
            "Chi2": chi_square(col)
        }
    else:
        drift_report[col] = {
            "KS_test": ks_test(col),
            "PSI": psi(col)
        }

# Exportar reporte
drift_df = pd.DataFrame(drift_report).T
os.makedirs("reports", exist_ok=True)
drift_df.to_csv("reports/drift_metrics.csv")
